# Project: Forecasting Aalborg temperature 🌡️
How warm will it be in Aalborg in 1, 3 or 7 days? With 7 years of hourly DMI observations, beat two classic baselines (*persistence*: same as today; *climatology*: the normal for that date) with the simplest real forecasting model: **climatology + a decaying anomaly**, i.e. an AR(1) on deviations from normal, the backbone of many classical models.

Topic: [[Classical Forecasting Models]], [[Time Series Forecasting]] · guide note: [[Project - Forecasting Aalborg Temperature]] · data: [DMI frie data](https://www.dmi.dk/frie-data)

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import json, pathlib, urllib.parse, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
f = pathlib.Path("data/dmi_06030_temp_2018_2024.csv"); f.parent.mkdir(exist_ok=True)
if not f.exists():
    q = dict(stationId="06030", parameterId="temp_mean_past1h", datetime="2018-01-01T00:00:00Z/2025-01-01T00:00:00Z", limit=300000)
    d = json.load(urllib.request.urlopen("https://opendataapi.dmi.dk/v2/metObs/collections/observation/items?" + urllib.parse.urlencode(q)))
    pd.DataFrame([x["properties"] for x in d["features"]])[["observed", "value"]].to_csv(f, index=False)
t = pd.read_csv(f); daily = t.assign(d=pd.to_datetime(t.observed)).set_index("d").value.sort_index().resample("D").mean().interpolate(limit=3).dropna()
daily.index = daily.index.tz_localize(None)
train, test = daily[:"2022-12-31"], daily["2023-01-01":]
print(f"{len(daily)} days at Flyvestation Aalborg; train {train.index.min():%Y}–{train.index.max():%Y}, test 2023–2024")
daily.plot(figsize=(12, 3), lw=.6, title="daily mean temperature, Aalborg (°C)"); plt.show()

## 1. Climatology
The 'normal' temperature per day of year: average the training data by `dayofyear` (1–366), fill gaps, then smooth with a **circular** 31-day centred rolling mean (wrap December into January). Return a Series indexed 1…366.

In [ ]:
def climatology(train):
    # TODO 1: day-of-year means, smoothed with a 31-day circular rolling window; index 1..366
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_clim():
    idx = pd.date_range("2001-01-01", "2004-12-31"); s = pd.Series(10 + 8 * np.sin(2 * np.pi * (idx.dayofyear - 100) / 365.25), idx)
    c = climatology(s)
    return len(c) == 366 and list(c.index[:2]) == [1, 2] and abs(c.loc[191] - 18) < 0.5 and abs(c.loc[9] - 2) < 0.5 and not c.isna().any()
check("climatology", _t_clim, "A pure sine with mean 10, amplitude 8 must give ≈ 18 in July and ≈ 2 in early January, with no NaNs.")

## 2. Climatology + decaying anomaly\nToday's anomaly $a = x_{\text{today}} - \text{clim}_{\text{today}}$ persists but fades: the forecast for $h$ days ahead is $\text{clim}_{\text{target}} + \phi^h a$. With $\phi=1$ it's persistence of the anomaly, with $\phi=0$ pure climatology.

In [ ]:
def forecast(last_value, clim_today, clim_target, h, phi):
    # TODO 2: climatology of the target day plus phi^h times today's anomaly
    raise NotImplementedError  # TODO 2

In [ ]:
check("forecast", lambda: forecast(5.0, 3.0, 4.0, 2, 0.5) == 4.5 and forecast(5.0, 3.0, 4.0, 1, 0.0) == 4.0, "Anomaly 2 decays to 0.5 after 2 days at phi 0.5 → 4.5.")
if ready("climatology", "forecast"):
    clim = climatology(train)
    anom = train - clim.loc[train.index.dayofyear].to_numpy()
    phi = float(np.corrcoef(anom[:-1], anom[1:])[0, 1]); print(f"lag-1 autocorrelation of anomalies (= fitted phi): {phi:.2f}")
    rows = {}
    for h in (1, 3, 7):
        x0 = test[:-h].to_numpy(); target = test[h:].to_numpy()
        c0 = clim.loc[test.index[:-h].dayofyear].to_numpy(); ch = clim.loc[test.index[h:].dayofyear].to_numpy()
        rows[f"{h} day(s)"] = {"persistence": np.mean(np.abs(target - x0)), "climatology": np.mean(np.abs(target - ch)),
                               "clim + AR(1) anomaly": np.mean(np.abs(target - forecast(x0, c0, ch, h, phi)))}
    res = pd.DataFrame(rows).T; print(res.round(2)); res.plot.bar(rot=0, figsize=(7, 3), title="test MAE (°C), 2023–2024"); plt.show()
    check("beats both baselines", lambda: all(res.loc[k, "clim + AR(1) anomaly"] < min(res.loc[k, "persistence"], res.loc[k, "climatology"]) for k in res.index),
          "The anomaly model should beat persistence and climatology at every horizon.")

**Read the table:** persistence wins short-term over climatology, climatology wins long-term, and the AR(1) anomaly model interpolates between them automatically. Weather services do the same with physics instead of $\phi$.

<details><summary>▶ Solution</summary>

```python
def climatology(train):
    doy = train.index.dayofyear
    clim = train.groupby(doy).mean()
    clim = clim.reindex(range(1, 367)).interpolate().bfill().ffill()
    smooth = pd.concat([clim.iloc[-15:], clim, clim.iloc[:15]]).rolling(31, center=True).mean().iloc[15:-15]
    smooth.index = range(1, 367)
    return smooth
```

```python
def forecast(last_value, clim_today, clim_target, h, phi):
    anomaly = last_value - clim_today
    return clim_target + (phi ** h) * anomaly
```
</details>

## Stretch goals
- Fit Holt–Winters / ETS with `statsmodels` (`pip install statsmodels`) on weekly data and compare.
- Use the anomaly of the last 3 days (AR(3)) – does it help at h = 3?
- Is 2023–2024 warmer than the 2018–2022 normal? Plot the anomaly's rolling 90-day mean.